# 2 · Vector layers

The vector builders draw a pyramids `FeatureCollection`: `points` (optionally coloured/sized by an
attribute), `lines` (line geometries), `polygons` (outlines or fills) and `choropleth` (polygons coloured by a
column). We use the real **Rhine-basin** layers — 34 discharge gauges, the river centre-lines and the
basin outline (all EPSG:3035, reprojected to the display CRS by pyramids).

**Setup** — backends and the three Rhine vector files.

In [ ]:
import warnings
warnings.filterwarnings("ignore")            # hide backend-mismatch style warnings in the docs

from pathlib import Path

# Resolve the repo root so the bundled sample data is found whether this runs from
# docs/examples/interactive/ (mkdocs) or the repository root.
ROOT = Path.cwd()
while not (ROOT / "examples" / "data" / "LisbonElevation.tif").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "examples" / "data"

import io
from IPython.display import Image
import matplotlib.pyplot as plt
import holoviews as hv
hv.extension("bokeh", "matplotlib")          # Bokeh = interactive; matplotlib = static (projection/barbs)

def show(m, static=False, dpi=96):
    """Display the map — **interactive Bokeh** by default (pan / zoom / hover).

    RUN THIS NOTEBOOK (Run All / Shift-Enter) to see the live interactive maps: a saved .ipynb only
    shows interactive plots once the viewer executes their JavaScript (a running kernel, or a trusted
    notebook). Pass ``static=True`` (used for the matplotlib-only layers — projection, streamlines,
    wind barbs, which have no Bokeh glyph) to render a static PNG instead.
    """
    if static or getattr(m, "_projection", None) is not None:
        fig = hv.render(m.render(), backend="matplotlib")
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
        plt.close(fig)
        return Image(data=buf.getvalue())
    return m.render()                        # interactive Bokeh object

from pyramids.feature import FeatureCollection
from digitalearth.interactive import InteractiveMap

gauges = FeatureCollection.read_file(str(DATA / "rhine_gauges.geojson"))
river  = FeatureCollection.read_file(str(DATA / "rhine_river_centerline.geojson"))
basin  = FeatureCollection.read_file(str(DATA / "rhine_basin.geojson"))

### `points` — coloured by an attribute
Colour the gauges by their `discharge` value and enlarge the markers. Hover is enabled by default in the live (Bokeh) map.

In [ ]:
m = InteractiveMap(crs=3857, title="Rhine gauges by discharge")
m.points(gauges, column="discharge", size=10, cmap="plasma")
show(m)

### `lines` — line features
`lines` draws line geometries — here the river centre-lines.

In [ ]:
m = InteractiveMap(crs=3857, title="Rhine river network")
m.lines(river, color="navy")
show(m)

### `polygons` — outlines
With no `column`, `polygons` draws outline-only (transparent fill) — good for showing an extent like the basin boundary.

In [ ]:
m = InteractiveMap(crs=3857, title="Rhine basin outline")
m.polygons(basin, line_color="firebrick")
show(m)

### `choropleth` — polygons coloured by a column
`choropleth` fills polygons by an attribute. We build visible polygons by buffering each gauge into an 8 km 'catchment' circle (EPSG:3035 is in metres) and colour them by `discharge`.

In [ ]:
catchments = gauges.copy()
catchments["geometry"] = catchments.geometry.buffer(8000.0)   # 8 km circles around each gauge

m = InteractiveMap(crs=3857, title="gauge catchments by discharge")
m.choropleth(catchments, "discharge", cmap="viridis")
show(m)

Layers **compose**: basin outline, river network and gauges in one overlay — a finished basin map.

In [ ]:
m = InteractiveMap(crs=3857, title="Rhine basin overview")
m.polygons(basin, line_color="grey")
m.lines(river, color="steelblue")
m.points(gauges, column="discharge", size=8, cmap="plasma")
show(m)